# Introduccion Clustering

# Importacion de librerias

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
%pip install --upgrade scipy statsmodels scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

# Librerias para clustering
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet
from scipy.spatial.distance import pdist, squareform


   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.5 MB 8.2 MB/s eta 0:00:01
   --------------- ------------------------ 3.7/9.5 MB 8.7 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.5 MB 9.9 MB/s eta 0:00:01
   ------------------------------------- -- 8.9/9.5 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 9.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.0 MB 7.2 MB/s eta 0:00:01
   -------------- ------------------------- 2.9/8.0 MB 7.5 MB/s eta 0:00:01
   ---------------------- ----------------- 4.5/8.0 MB 7.2 MB/s eta 0:00:01
   ------------------------------ --------- 6.0/8.0 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.0 MB 7.5 MB/s eta 0:00:01
   -----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.


---

# EDA

In [8]:
# Load wine.data and convert to CSV with proper column names
column_names = ['Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium', 
                'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins',
                'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline']

df_wine = pd.read_csv('../../../../Archivos-Analisis/files-practica-m45/wine.data', names=column_names)
df_wine.to_csv('../../../../Archivos-Analisis/files-practica-m45/wine.csv', index=False)
df_wine.head()

,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [9]:
df_wine.shape

(178, 13)

In [10]:
df_wine.dtypes

Alcohol                         float64
Malic acid                      float64
Ash                             float64
Alcalinity of ash               float64
Magnesium                         int64
Total phenols                   float64
Flavanoids                      float64
Nonflavanoid phenols            float64
Proanthocyanins                 float64
Color intensity                 float64
Hue                             float64
OD280/OD315 of diluted wines    float64
Proline                           int64
dtype: object

In [11]:
df_wine.info()

<class 'pandas.core.frame.DataFrame'>
Index: 178 entries, 1 to 3
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Alcohol                       178 non-null    float64
 1   Malic acid                    178 non-null    float64
 2   Ash                           178 non-null    float64
 3   Alcalinity of ash             178 non-null    float64
 4   Magnesium                     178 non-null    int64  
 5   Total phenols                 178 non-null    float64
 6   Flavanoids                    178 non-null    float64
 7   Nonflavanoid phenols          178 non-null    float64
 8   Proanthocyanins               178 non-null    float64
 9   Color intensity               178 non-null    float64
 10  Hue                           178 non-null    float64
 11  OD280/OD315 of diluted wines  178 non-null    float64
 12  Proline                       178 non-null    int64  
dtypes: float64(1

In [14]:
df_wine.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Alcohol,178.0,13.00,0.81,11.03,12.36,13.05,13.68,14.83
Malic acid,178.0,2.34,1.12,0.74,1.60,1.87,3.08,5.80
Ash,178.0,2.37,0.27,1.36,2.21,2.36,2.56,3.23
Alcalinity of ash,178.0,19.49,3.34,10.60,17.20,19.50,21.50,30.00
Magnesium,178.0,99.74,14.28,70.00,88.00,98.00,107.00,162.00
Total phenols,178.0,2.30,0.63,0.98,1.74,2.36,2.80,3.88
Flavanoids,178.0,2.03,1.00,0.34,1.20,2.13,2.88,5.08
Nonflavanoid phenols,178.0,0.36,0.12,0.13,0.27,0.34,0.44,0.66
Proanthocyanins,178.0,1.59,0.57,0.41,1.25,1.56,1.95,3.58
Color intensity,178.0,5.06,2.32,1.28,3.22,4.69,6.20,13.00


In [15]:
df_wine.isnull().sum()

Alcohol                         0
Malic acid                      0
Ash                             0
Alcalinity of ash               0
Magnesium                       0
Total phenols                   0
Flavanoids                      0
Nonflavanoid phenols            0
Proanthocyanins                 0
Color intensity                 0
Hue                             0
OD280/OD315 of diluted wines    0
Proline                         0
dtype: int64

In [19]:
df_wine.nunique().sort_values()

Nonflavanoid phenols             39
Magnesium                        53
Alcalinity of ash                63
Hue                              78
Ash                              79
Total phenols                    97
Proanthocyanins                 101
Proline                         121
OD280/OD315 of diluted wines    122
Alcohol                         126
Flavanoids                      132
Color intensity                 132
Malic acid                      133
dtype: int64

In [ ]:
# Cuenta el numero de 0 por cada columna
# Esto se hizo por que en algunas de ellas el minimo es cero 
df_wine[df_wine==0].count() / df_wine.count()

Alcohol                         0.0
Malic acid                      0.0
Ash                             0.0
Alcalinity of ash               0.0
Magnesium                       0.0
Total phenols                   0.0
Flavanoids                      0.0
Nonflavanoid phenols            0.0
Proanthocyanins                 0.0
Color intensity                 0.0
Hue                             0.0
OD280/OD315 of diluted wines    0.0
Proline                         0.0
dtype: float64

In [ ]:
# Buscamos duplcados
df_wine.duplicated()

# Numero de duplicados
df_wine.duplicated().sum()

np.int64(0)

Insights
- No existen columnas con valores de 0
- Tampoco se tienen columnas con indicadores tipo ID, por lo que ne deben eliminar
- Ademas de que todas las columnas cuentan con multiples datos unicos, por lo que tampoco se deben de eliminar en caso de que llegaran a tener 1 unico valor
- No se encontraron valores nulos en ninguna de las columnas por lo que no se requerira instanciar informacion
- De igual forma no se encontraron valores duplicados

---

# Analisis Bivariado y Correlacional

## Correlacional

## Bivariado